<a href="https://colab.research.google.com/github/SalamaSobhy/salama/blob/main/skin_cancer_model_keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub

In [2]:
import kagglehub
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, save_img

# ===============================
# LOAD DATASET (READ ONLY)
# ===============================
path = kagglehub.dataset_download("fanconic/skin-cancer-malignant-vs-benign")
SOURCE_DIR = path + "/train"

# ===============================
# CREATE OUTPUT DIR (WRITABLE)
# ===============================
OUTPUT_DIR = "/content/augmented_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "benign"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "malignant"), exist_ok=True)

print("Saving augmented images to:", OUTPUT_DIR)

# ===============================
# AUGMENTATION (خفيف طبيًا)
# ===============================
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

# ===============================
# FUNCTION
# ===============================
def augment_and_save(class_name, save_count=3):
    src_path = os.path.join(SOURCE_DIR, class_name)
    dst_path = os.path.join(OUTPUT_DIR, class_name)

    images = os.listdir(src_path)

    for img_name in images:
        img_path = os.path.join(src_path, img_name)
        img = load_img(img_path, target_size=(224, 224))
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)

        i = 0
        for batch in datagen.flow(x, batch_size=1):
            save_img(
                os.path.join(dst_path, f"aug_{i}_{img_name}"),
                batch[0]
            )
            i += 1
            if i >= save_count:
                break

# ===============================
# APPLY AUGMENTATION
# ===============================
augment_and_save("benign", save_count=3)
augment_and_save("malignant", save_count=3)

print("Augmentation completed successfully")

100%|██████████| 325M/325M [00:02<00:00, 131MB/s]

Extracting files...


Saving augmented images to: /content/augmented_data
Augmentation completed successfully


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# ===============================
# DATA PATH (AUGMENTED DATA)
# ===============================
DATA_DIR = "/content/augmented_data"
print("Using dataset from:", DATA_DIR)

# ===============================
# PARAMETERS
# ===============================
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 60

# ===============================
# DATA GENERATOR (NO MORE AUGMENTATION)
# ===============================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

val_data = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

# ===============================
# MODEL
# ===============================
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.4),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# ===============================
# TRAIN
# ===============================
model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)


Using dataset from: /content/augmented_data
Found 6329 images belonging to 2 classes.
Found 1582 images belonging to 2 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/60
396/396 ━━━━━━━━━━━━━━━━━━━━ 319s 786ms/step - accuracy: 0.7499 - loss: 0.5063 - val_accuracy: 0.8161 - val_loss: 0.3953
Epoch 2/60
396/396 ━━━━━━━━━━━━━━━━━━━━ 333s 841ms/step - accuracy: 0.8173 - loss: 0.3964 - val_accuracy: 0.8350 - val_loss: 0.3591
Epoch 3/60
396/396 ━━━━━━━━━━━━━━━━━━━━ 317s 801ms/step - accuracy: 0.8385 - loss: 0.3532 - val_accuracy: 0.8407 - val_loss: 0.3449
Epoch 4/60
396/396 ━━━━━━━━━━━━━━━━━━━━ 325s 820ms/step - accuracy: 0.8599 - loss: 0.3275 - val_accuracy: 0.8571 - val_loss: 0.3167
Epoch 5/60
396/396 ━━━━━━━━━━━━━━━━━━━━ 298s 752ms/step - accuracy: 0.8676 - loss: 0.3064 - val_accuracy: 0.8666 - val_loss: 0.3029
Epoch 6/60
396/396 ━━━━━━━━━━━━━━━━━━━━ 308s 779ms/step - accuracy: 0.8738 - loss: 0.2873 - val_accuracy: 0.8767 - val_loss: 0.2946
Epoch 7/60
396/396 ━━━━━━━━━━━━

In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
model.save("/content/drive/MyDrive/skin_cancer_model.keras")


In [6]:
model.save("skin_cancer_model.keras")


In [7]:
import os

size_bytes = os.path.getsize("skin_cancer_model.keras")
size_mb = size_bytes / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")


Model size: 11.06 MB
